# 02 — Nettoyage des données
Lit directement le CSV brut → génère `data/clean.csv`

In [ ]:
import csv
import pandas as pd
import numpy as np
from pathlib import Path

BASE     = Path('d:/Smart-Free_GestionMess+Feedback/Smart-Free_GestionMess/ml-service')
CSV_PATH = BASE / 'freelancer_job_postings.csv'
DATA_DIR = BASE / 'data'
DATA_DIR.mkdir(exist_ok=True)

COLS = ['projectId','job_title','job_description','tags','client_state',
        'client_country','client_average_rating','client_review_count',
        'min_price','max_price','avg_price','currency','rate_type']

rows = []
with open(CSV_PATH, encoding='utf-8', errors='replace') as f:
    for i, raw in enumerate(f):
        if i == 0: continue
        line = raw.strip()
        while line.endswith(';'): line = line[:-1]
        if line.startswith('"') and line.endswith('"'):
            line = line[1:-1].replace('""', '"')
        parts = list(csv.reader([line]))[0]
        if len(parts) >= 13:
            rows.append(parts[:13])

df = pd.DataFrame(rows, columns=COLS)
print('Shape brut :', df.shape)
df.head(2)

In [1]:
num_cols = ['client_average_rating','client_review_count','min_price','max_price','avg_price']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

before = len(df)
df = df.dropna(subset=['avg_price','min_price','max_price'])
print(f'Lignes supprimées (prix NaN) : {before - len(df)} | Reste : {len(df)}')

NameError: name 'pd' is not defined

In [ ]:
Q1, Q3 = df['avg_price'].quantile(0.01), df['avg_price'].quantile(0.99)
before = len(df)
df = df[(df['avg_price'] >= Q1) & (df['avg_price'] <= Q3)]
print(f'Outliers supprimés : {before - len(df)} | bornes [{Q1:.0f}, {Q3:.0f}] | Reste : {len(df)}')

In [ ]:
def clean_tags(raw):
    if pd.isna(raw): return ''
    s = str(raw).strip("[]").replace("'", "").replace('"', '')
    return ','.join(t.strip().lower() for t in s.split(',') if t.strip())

df['tags_clean'] = df['tags'].apply(clean_tags)

df['rate_type'] = df['rate_type'].str.strip().str.lower()
df = df[df['rate_type'].isin(['hourly','fixed'])]

df['client_average_rating'] = df['client_average_rating'].fillna(df['client_average_rating'].median())
df['client_review_count']   = df['client_review_count'].fillna(0).astype(int)
df['client_country']        = df['client_country'].fillna('Unknown')
df['client_state']          = df['client_state'].fillna('Unknown')

print('Shape final :', df.shape)
print('NaN restants :', df.isnull().sum().sum())

In [ ]:
out = DATA_DIR / 'clean.csv'
df.to_csv(out, index=False)
print(f'Sauvegarde OK : {out}')
df[['projectId','job_title','tags_clean','avg_price','rate_type']].head(5)